In [11]:
%pip install --upgrade transformers accelerate tokenizers datasets scikit-learn torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import transformers
import tokenizers
import accelerate
import torch

print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("accelerate:", accelerate.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

transformers: 5.14.1
tokenizers: 0.22.2
accelerate: 1.14.0
torch: 2.13.0+cpu
CUDA available: False


In [2]:
import pandas as pd

train = pd.read_csv("../data/splits/train.csv").fillna("")
val = pd.read_csv("../data/splits/val.csv").fillna("")
test = pd.read_csv("../data/splits/test.csv").fillna("")

print(train.shape, val.shape, test.shape)

(34996, 4) (7501, 4) (7500, 4)


In [3]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
sample_texts = train["clean_content"].head(3).tolist()
encoded = tokenizer(sample_texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
print(encoded["input_ids"].shape)

torch.Size([3, 163])


In [5]:
small_train = train.sample(n=300, random_state=42).reset_index(drop=True)
small_val = val.sample(n=100, random_state=42).reset_index(drop=True)

train_encodings = tokenizer(small_train["clean_content"].tolist(), padding=True, truncation=True, max_length=256, return_tensors="pt")
val_encodings = tokenizer(small_val["clean_content"].tolist(), padding=True, truncation=True, max_length=256, return_tensors="pt")

print(train_encodings["input_ids"].shape)
print(val_encodings["input_ids"].shape)

torch.Size([300, 250])
torch.Size([100, 235])


In [6]:
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = ReviewDataset(train_encodings, small_train["label"].tolist())
val_dataset = ReviewDataset(val_encodings, small_val["label"].tolist())

print(len(train_dataset), len(val_dataset))

300 100


In [7]:
from transformers import Trainer, TrainingArguments

test_args = TrainingArguments(
    output_dir="../models/test_run",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_steps=5,
    report_to="none",
)

trainer = Trainer(model=model, args=test_args, train_dataset=train_dataset, eval_dataset=val_dataset)
trainer.train()

c:\Users\tanis\OneDrive\Desktop\insight-agent\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.408158,0.404237


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\tanis\OneDrive\Desktop\insight-agent\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=38, training_loss=0.5730107332530775, metrics={'train_runtime': 439.7574, 'train_samples_per_second': 0.682, 'train_steps_per_second': 0.086, 'total_flos': 19404404100000.0, 'train_loss': 0.5730107332530775, 'epoch': 1.0})